# Task 4 — Join & Transformation
## 1. Load Validated Stock Data

In [8]:
import pandas as pd
from pathlib import Path

In [9]:
# Load the validated stock data
interim_folder = Path("../data/interim")
validated_file = interim_folder / "validated.csv"

stock_df = pd.read_csv(
    validated_file,
    dtype={"symbol": "string"},
    parse_dates=["date"]
)

stock_df.head()

,symbol,date,open,high,low,close,adj_close,volume
0,1120,2013-01-01,16.4380,16.5630,16.3130,16.3755,10.5869,7578353
1,1120,2013-01-02,16.3755,16.7505,16.3755,16.6880,10.7890,9374450
2,1120,2013-01-03,61.6173,61.6173,61.6173,61.6173,39.8362,0
3,1120,2013-01-06,16.8755,16.8755,16.8130,16.8130,10.8698,4490083
4,1120,2013-01-07,16.8130,16.8755,16.6255,16.8755,10.9102,11432063


In [10]:
# Create year, quarter, and a common quarter key
stock_df["year"] = stock_df["date"].dt.year
stock_df["quarter"] = "Q" + stock_df["date"].dt.quarter.astype(str)
stock_df["year_quarter"] = (
    stock_df["year"].astype(str) + "-" + stock_df["quarter"]
)

stock_df[["symbol", "date", "year", "quarter", "year_quarter"]].head()

,symbol,date,year,quarter,year_quarter
0,1120,2013-01-01,2013,Q1,2013-Q1
1,1120,2013-01-02,2013,Q1,2013-Q1
2,1120,2013-01-03,2013,Q1,2013-Q1
3,1120,2013-01-06,2013,Q1,2013-Q1
4,1120,2013-01-07,2013,Q1,2013-Q1


In [11]:
# Aggregate daily stock data to quarterly level
stock_quarterly = (
    stock_df
    .sort_values(["symbol", "date"])
    .groupby(["symbol", "year", "quarter", "year_quarter"])
    .agg(
        quarter_open=("open", "first"),
        quarter_high=("high", "max"),
        quarter_low=("low", "min"),
        quarter_close=("close", "last"),
        avg_close=("close", "mean"),
        total_volume=("volume", "sum")
    )
    .reset_index()
)

stock_quarterly.head()

,symbol,year,quarter,year_quarter,quarter_open,quarter_high,quarter_low,quarter_close,avg_close,total_volume
0,1120,2013,Q1,2013-Q1,16.4380,65.5405,16.0005,16.4380,19.905837,381559706
1,1120,2013,Q2,2013-Q2,16.4380,71.5000,16.0630,18.0005,17.968083,394491300
2,1120,2013,Q3,2013-Q3,18.0005,20.3131,18.0005,19.3131,19.265183,349130787
3,1120,2013,Q4,2013-Q4,19.2506,71.7714,18.0005,18.2506,19.597710,547883090
4,1120,2014,Q1,2014-Q1,18.2506,18.8131,17.3755,18.6881,18.051317,752506574


In [12]:
# Check the number of quarterly stock records
print("Quarterly rows:", len(stock_quarterly))
print("Expected rows:", 15 * 53)

Quarterly rows: 795
Expected rows: 795


In [13]:
# Find missing company-quarter combinations

all_quarters = stock_df["year_quarter"].unique()
all_symbols = stock_df["symbol"].unique()

expected = pd.MultiIndex.from_product(
    [all_symbols, all_quarters],
    names=["symbol", "year_quarter"]
)

actual = pd.MultiIndex.from_frame(
    stock_quarterly[["symbol", "year_quarter"]]
)

missing = expected.difference(actual)

print("Missing company-quarters:", len(missing))
print(missing)
stock_df[stock_df["symbol"] == "1180"]["date"].agg(["min", "max"])

Missing company-quarters: 0
MultiIndex([], names=['symbol', 'year_quarter'])


min   NaT
max   NaT
Name: date, dtype: datetime64[s]

In [14]:
# Check for duplicate company-quarter combinations
stock_quarterly.duplicated(
    subset=["symbol", "year_quarter"]
).sum()

np.int64(0)

In [15]:
# Check for missing values after transformation
stock_quarterly.isnull().sum()

symbol           0
year             0
quarter          0
year_quarter     0
quarter_open     0
quarter_high     0
quarter_low      0
quarter_close    0
avg_close        0
total_volume     0
dtype: int64

In [16]:
# Save the quarterly stock data
stock_quarterly.to_csv(
    interim_folder / "stock_quarterly.csv",
    index=False
)

print("Saved:", interim_folder / "stock_quarterly.csv")

Saved: ..\data\interim\stock_quarterly.csv


## 2. Stock Transformation Rules

The daily stock data was aggregated to quarterly level.

- `quarter_open`: first opening price in the quarter
- `quarter_high`: highest price during the quarter
- `quarter_low`: lowest price during the quarter
- `quarter_close`: last closing price in the quarter
- `avg_close`: average closing price during the quarter
- `total_volume`: total trading volume during the quarter

## 3. Stock Data Grain

The quarterly stock dataset has a grain of one row per company per quarter.
The daily stock data is preserved in `validated.csv`, while `stock_quarterly.csv` is used for the quarterly integration with the currency and economic data.

In [18]:
# Check that the quarterly stock data has the expected grain
assert len(stock_quarterly) == 795
assert stock_quarterly.duplicated(
    subset=["symbol", "year_quarter"]
).sum() == 0

print("Stock quarterly transformation passed.")

Stock quarterly transformation passed.


In [19]:
# Check the quarterly stock dataset
stock_quarterly.shape

(795, 10)